In [1]:
import numpy as np
import pandas as pd
import concurrent.futures
import timeit
from functools import partial
from copy import deepcopy
import sys
from IPython.display import display
import os
import pprint
# Add measurement mcts python package to path
sys.path.append('../src/measurement_mcts')
from measurement_mcts.mcts.mcts import mcts_with_rollout
from measurement_mcts.mcts.tree_viz import render_pyvis
from measurement_mcts.state_evaluation.hertg import HERTG
from measurement_mcts.environment.measurement_control_env import MeasurementControlEnvironment
from measurement_mcts.utils.metrics import get_percent_done, save_environment_config, get_mcts_metrics, worker_wrapper

# Create the environment
env = MeasurementControlEnvironment(init_reset=False)

Toy Measurement Control Initialized


In [2]:
# Save the object configurations to a folder for use in the experiment
num_trials = 100
folder = "trial_configs1"

# Create the folder if it doesn't exist
if not os.path.exists(folder):
    os.makedirs(folder)

for i in range(num_trials):
    env.reset() # Reset the environment to a new random state
    env.save_state(folder, f"trial_{i}") # Save the environment state to a file
    
# Save the environment configuration
save_environment_config(env, folder)

Environment configuration saved to trial_configs1/env_config.txt


In [ ]:
# Load the environment state once.
state = env.load_state("trial_configs1", "trial_0")
state = env.get_state()

def run_trial(trial):
    print(f'Running trial {trial}')
    return get_mcts_metrics(env, state)

all_results = []
with concurrent.futures.ProcessPoolExecutor() as executor:
    futures = []
    # Schedule a job for each trial.
    for trial in range(20):
        futures.append(executor.submit(run_trial, trial))
    
    # Gather the results as tasks complete.
    for future in concurrent.futures.as_completed(futures):
        try:
            result = future.result()
            all_results.append(result)
        except Exception as e:
            print("An error occurred during execution:", e)

df = pd.DataFrame(all_results)
display(df)


In [ ]:
# Run the same ennvironment multiple times and see if results are the same
state = env.load_state("trial_configs1", "trial_0")
state = env.get_state()
metrics = []
for i in range(5):
    print(f'Running trial {i}')
    metrics.append(get_mcts_metrics(env, state))
    
df = pd.DataFrame(metrics)
display(df)

Running trial 0


c:\Users\austi\Documents\CodeScratch\MeasurementMCTS\metrics\../src/measurement_mcts\measurement_mcts\mcts\mcts.py:222: RuntimeWarning: divide by zero encountered in log
  * np.sqrt(np.log(self.number_visits)


KeyboardInterrupt: 

In [ ]:
def run_experiments_for_rollout_methods(rollout_methods, trial_config_path, trial_config_names, num_same_config=10, **kwargs):
    """
    For each rollout method, run experiments for every pre-saved trial configuration.
    
    Parameters:
        rollout_methods (list of str): List of rollout methods to test.
        trial_config_path (str): Directory containing the trial configuration files.
        trial_config_names (list of str): List of trial configuration file names (without the .pkl extension).
        kwargs: Additional keyword arguments to pass to the worker_wrapper.
    
    Returns:
        List[dict]: A list of dictionaries with the metrics for each run.
    """
    all_results = []
    with concurrent.futures.ProcessPoolExecutor() as executor:
        futures = []
        # Schedule a job for each combination of rollout method and trial configuration.
        for method in rollout_methods:
            for config_name in trial_config_names:
                futures.append(
                    executor.submit(
                        worker_wrapper,
                        trial_config_name=config_name,
                        rollout_method=method,
                        trial_config_path=trial_config_path,
                        **kwargs
                    )
                )
        # Gather the results as tasks complete.
        for future in concurrent.futures.as_completed(futures):
            try:
                result = future.result()
                all_results.append(result)
            except Exception as e:
                print("An error occurred during execution:", e)
    return all_results

def run_experiments_for_horizon_lengths(horizon_lengths, trial_config_path, trial_config_names, **kwargs):
    """
    For each horizon length, run experiments for every pre-saved trial configuration.
    
    Parameters:
        horizon_lengths (list of int): List of horizon lengths to test.
        trial_config_path (str): Directory containing the trial configuration files.
        trial_config_names (list of str): List of trial configuration file names (without the .pkl extension).
        kwargs: Additional keyword arguments to pass to the worker_wrapper.
    
    Returns:
        List[dict]: A list of dictionaries with the metrics for each run.
    """
    all_results = []
    with concurrent.futures.ProcessPoolExecutor() as executor:
        futures = []
        # Schedule a job for each combination of horizon length and trial configuration.
        for horizon in horizon_lengths:
            for config_name in trial_config_names:
                futures.append(
                    executor.submit(
                        worker_wrapper,
                        trial_config_name=config_name,
                        horizon_length=horizon,
                        trial_config_path=trial_config_path,
                        **kwargs
                    )
                )
        # Gather the results as tasks complete.
        for future in concurrent.futures.as_completed(futures):
            try:
                result = future.result()
                all_results.append(result)
            except Exception as e:
                print("An error occurred during execution:", e)
    return all_results


# Define different experiment configurations
rollout_methods = ['random', 'same', 'random_same', 'zero', 'accelerate']
horizon_lengths = [1, 2, 4, 6, 8, 10, 12, 14]
learning_iterations = [50, 100, 200, 400, 600, 800, 1000]

# Path to the directory where trial configurations are stored.
trial_config_path = "trial_configs1"  # Update this path as needed.

# Get list of pickle files in folder
trial_config_names = os.listdir(trial_config_path)
trial_config_names = [f for f in trial_config_names if f.endswith('.pkl')]
trial_config_names = [f[:-4] for f in trial_config_names] # Remove .pkl extension

# Run experiments for each rollout method over all the pre-generated configurations.
results = run_experiments_for_rollout_methods(
    rollout_methods,
    trial_config_path,
    trial_config_names,
    max_actions=200,
    LI=100,
    EF=0.1,
    DF=1.0,
    hertg_method='static',
    rollout_pre_collision_stop=True
)

# Compile individual run results into a DataFrame and save to a CSV file.
df = pd.DataFrame(results)
print("Individual run metrics:")
print(f'Number of runs: {len(df)}')
display(df.head())
df.to_csv("mcts_metrics_same_states_100trials.csv", index=False)


Toy Measurement Control InitializedToy Measurement Control Initialized
Toy Measurement Control InitializedToy Measurement Control InitializedToy Measurement Control InitializedToy Measurement Control InitializedToy Measurement Control InitializedToy Measurement Control Initialized

Toy Measurement Control InitializedToy Measurement Control InitializedToy Measurement Control Initialized
Toy Measurement Control Initialized

Toy Measurement Control Initialized
Toy Measurement Control InitializedToy Measurement Control InitializedToy Measurement Control InitializedToy Measurement Control InitializedToy Measurement Control Initialized



/home/austin/MeasurementMCTS/metrics/../src/measurement_mcts/measurement_mcts/mcts/mcts.py:222: RuntimeWarning: divide by zero encountered in log
  * np.sqrt(np.log(self.number_visits)


Toy Measurement Control Initialized


/home/austin/MeasurementMCTS/metrics/../src/measurement_mcts/measurement_mcts/mcts/mcts.py:222: RuntimeWarning: divide by zero encountered in log
  * np.sqrt(np.log(self.number_visits)




Toy Measurement Control Initialized



...

/home/austin/MeasurementMCTS/metrics/../src/measurement_mcts/measurement_mcts/mcts/mcts.py:222: RuntimeWarning: divide by zero encountered in log
  * np.sqrt(np.log(self.number_visits)


/home/austin/MeasurementMCTS/metrics/../src/measurement_mcts/measurement_mcts/mcts/mcts.py:222: RuntimeWarning: divide by zero encountered in log
  * np.sqrt(np.log(self.number_visits)


Toy Measurement Control InitializedToy Measurement Control Initialized



/home/austin/MeasurementMCTS/metrics/../src/measurement_mcts/measurement_mcts/mcts/mcts.py:222: RuntimeWarning: divide by zero encountered in log
  * np.sqrt(np.log(self.number_visits)
/home/austin/MeasurementMCTS/metrics/../src/measurement_mcts/measurement_mcts/mcts/mcts.py:222: RuntimeWarning: divide by zero encountered in log
  * np.sqrt(np.log(self.number_visits)


[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
sum_traces: 0.0
Toy Measurement Control Initialized
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
sum_traces: 0.0
Toy Measurement Control Initialized
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
sum_traces: 0.0
Toy Measurement Control Initialized
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
sum_traces: 0.0
Toy Measurement Control Initialized
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
sum_traces: 0.0
Toy Measurement Control Initialized
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
sum_traces: 0.0
Toy Measurement Control Initialized
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
sum_traces: 0.0
Toy Measurement Control Initialized
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
sum_traces: 0.0
Toy Measurement Control Initialized
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
sum_traces: 0.0
Toy Measurement Control Initialized
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
sum_traces: 0.0
Toy Measurement Control Initialized
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
sum_traces: 0.0
Toy Measurement Control Initialized
[0. 0. 0. 